# Module 4: LangSmith — Observability & Evaluations

> Part of the **Modular Workshops** series. Standalone, ~45 min.

We put the **commodities trading copilot** under observability. Using the desk-analyst agent from Module 2, we cover three parts plus a closing loop:

1. **Tracing** — generate traces with the trading copilot from Module 2, then query them with `list_runs` + filters.
2. **Offline evaluations** — build a dataset of desk tasks, score the copilot end-to-end (final-response with LLM-as-judge) and step-by-step (trajectory).
3. **Online evaluations** — score every new trace as it lands. Programmatic version + UI workflow.

Then **annotation queues** close the loop: route runs flagged by eval scores to a desk analyst for review.

> **Disclaimer:** The commodities examples in this workshop are illustrative only — not trading advice.

<img src="../images/evals-conceptual.png" style="width: auto; max-height: 400px; border-radius: 8px;">


## Setup


In [1]:
import sys
import logging
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

from utils.models import model
from utils.langsmith_rules import (
    get_or_create_annotation_queue,
    create_run_rule,
    delete_run_rule,
)

import os, time
from datetime import datetime, timedelta, timezone
from typing_extensions import TypedDict
from langchain_core.messages import SystemMessage, HumanMessage
from langsmith import Client, uuid7

# LangSmith's tracer logs benign "No indexed run ID" warnings when the copilot
# runs tools/subagents on parallel threads (seen in the tracing/eval cells).
# Traces still upload fine and execution is unaffected. Filter ONLY that
# message so real callback warnings stay visible.
class _NoIndexedRunIDFilter(logging.Filter):
    def filter(self, record):
        return "No indexed run ID" not in record.getMessage()

logging.getLogger("langchain_core.callbacks.manager").addFilter(_NoIndexedRunIDFilter())

client = Client()
print("LANGSMITH_TRACING:", os.environ.get("LANGSMITH_TRACING", "not set"))
print("Project:", os.environ.get("LANGSMITH_PROJECT", "default"))


LANGSMITH_TRACING: true
Project: trading-agent-workshop-william-reinhard


## Warm-up: Generate a few traces

Before we look at tracing and querying, let's actually produce some traces. 
We invoke the trading copilot from Module 2 (`agents/research_agent.py`) three times with **deliberately light** prompts — 
each one asks a quick market question and says "search at most once" so the runs finish in a few seconds instead of a few minutes.

On the trial run while building this module the warm-up took **~9 seconds total (3.1s avg per call)**. Expect similar.


In [2]:
from agents.research_agent import build_research_agent

agent = build_research_agent()

warmup_prompts = [
    "In one sentence, what is driving corn futures right now? Search at most once.",
    "In one sentence, what is the current outlook for soybean exports? Search at most once.",
    "In one sentence, what is moving edible oil prices this week? Search at most once.",
]

total = 0.0
for q in warmup_prompts:
    cfg = {"configurable": {"thread_id": str(uuid7())}}
    t0 = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=cfg)
    elapsed = time.perf_counter() - t0
    total += elapsed
    print(f"[{elapsed:4.1f}s] {result['messages'][-1].content[:120]}")

print(f"\nTotal: {total:.1f}s ({total/len(warmup_prompts):.1f}s avg)")


[ 4.2s] Corn futures are currently driven mainly by changes in ethanol demand tied to gasoline prices and government policies, t
[ 1.8s] The current outlook for soybean exports is positive, with U.S. soybean export forecasts for marketing year 2026/27 raise
[ 2.3s] Edible oil prices this week are being driven by supply disruptions from the Russia-Ukraine war, drought in Argentina, In

Total: 8.4s (2.8s avg)


## Part 1. Tracing + Querying Traces

Set `LANGSMITH_TRACING=true` and every LLM call, tool call, and state transition from the trading copilot lands in your tracing project — no code changes required. 
(The warm-up above already produced traces; this section pulls them back out.)

We use `client.list_runs(...)` to query them.


In [3]:
project_name = os.environ.get("LANGSMITH_PROJECT", "modular-workshops")
try:
    project = client.read_project(project_name=project_name)
    print(f"Project: {project.name}")
    print(f"View traces: {project.url}")
except Exception as e:
    print(f"Could not read project (this is fine if first run): {e}")


Project: trading-agent-workshop-william-reinhard
View traces: https://smith.langchain.com/o/e3272f09-bf53-4cd5-a37c-93bd272d3078/projects/p/156d5738-eb06-473c-8924-f83f59eea029


### 1.1 Pull recent traces

Useful filters on `client.list_runs(...)`:

- `project_name=` — scope to one project
- `start_time=` / `end_time=` — time window
- `run_type=` — `"llm"`, `"tool"`, `"chain"`, `"retriever"`
- `error=True` — only failed runs
- `is_root=True` — only top-level traces (not their children)
- `filter=` — LangSmith filter DSL (latency, feedback, attributes...)


In [4]:
from datetime import datetime, timedelta, timezone

# Pull the last hour of root traces from this workshop's project
since = datetime.now(timezone.utc) - timedelta(hours=1)

recent_runs = list(client.list_runs(
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    start_time=since,
    is_root=True,
    limit=20,
))

print(f"Found {len(recent_runs)} root run(s) in the last hour\n")
for r in recent_runs[:5]:
    latency = (r.end_time - r.start_time).total_seconds() if r.end_time else None
    print(f"- {r.id}  {r.name:25s}  latency={latency}s  error={r.error is not None}")


Found 13 root run(s) in the last hour

- 019f71e1-cae6-7910-bc34-a558e4fcdad8  LangGraph                  latency=Nones  error=False
- 019f71e1-c3ae-7240-b309-5353b698efd7  LangGraph                  latency=1.847549s  error=False
- 019f71e1-b323-74a2-950f-252a241f42f2  LangGraph                  latency=4.233174s  error=False
- 019f71d3-49f6-7cb0-ad82-1d4364b7e499  LangGraph                  latency=3.63203s  error=False
- 019f71d3-4062-7d41-a5cb-e44f6192dba2  LangGraph                  latency=2.45072s  error=False


### 1.2 Filter DSL — find slow or errored runs

The `filter` argument is a small expression language. Common patterns for auditing desk runs:

- `gt(latency, 5)` — slower than 5 seconds
- `eq(status, "error")` — failed runs
- `and(eq(feedback_key, "correctness"), lt(feedback_score, 0.5))` — low-scored desk answers on a feedback key
- Combine with `and(...)` / `or(...)`


In [5]:
# Find slow root runs in the last hour (>5s latency)
slow_runs = list(client.list_runs(
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    start_time=since,
    is_root=True,
    filter='gt(latency, 5)',
    limit=20,
))

print(f"{len(slow_runs)} slow root run(s) (>5s) in the last hour")
for r in slow_runs[:5]:
    latency = (r.end_time - r.start_time).total_seconds()
    print(f"  {r.name:25s}  {latency:.1f}s  {r.id}")


2 slow root run(s) (>5s) in the last hour
  LangGraph                  7.1s  019f71d3-24d3-7ba3-930e-3e2c748279df
  LangGraph                  7.6s  019f71be-a100-70a0-9026-621e8fe736ef


## Part 2. Offline Evaluations

**Offline evals** are the experiments you run on demand against a fixed dataset. 
Build a dataset once, score your agent against it whenever you change a prompt, a model, or a tool — get a clean before/after comparison.

Three pieces:
1. **Dataset** — labeled `inputs` + expected `outputs`
2. **Target function** — runs your agent on each example
3. **Evaluators** — score the output (LLM-as-judge or code-based)


### 2.1 Dataset

A small set of desk tasks a trading analyst would ask the copilot. Same input set, two reference shapes — one for final-response judging, one for trajectory matching.


In [6]:
examples = [
    {
        "inputs": {"query": "Write a one-line morning market note on corn to /corn_note.txt"},
        "outputs": {
            "reference_answer": "A one-line morning market note on corn should be saved to /corn_note.txt",
            "trajectory": ["write_file"],
        },
    },
    {
        "inputs": {"query": "Research the latest soybean export demand and write a brief desk note to /soybean_note.md"},
        "outputs": {
            "reference_answer": "A short desk note on recent soybean export demand, written to /soybean_note.md",
            "trajectory": ["task", "write_file"],
        },
    },
    {
        "inputs": {"query": "Write a short trade idea memo on edible oils to /oils_memo.md, then read it back to confirm"},
        "outputs": {
            "reference_answer": "A short edible-oils trade idea memo written to /oils_memo.md and read back",
            "trajectory": ["write_file", "read_file"],
        },
    },
    {
        "inputs": {"query": "Plan out a short research brief on the wheat market, then research it and write the brief to /wheat_brief.md"},
        "outputs": {
            "reference_answer": "A planned-out wheat market research brief written to /wheat_brief.md",
            "trajectory": ["write_todos", "task", "write_file"],
        },
    },
]

from utils.workshop import scoped
dataset_name = scoped("modular-workshops-evals")

if client.has_dataset(dataset_name=dataset_name):
    existing = client.read_dataset(dataset_name=dataset_name)
    client.delete_dataset(dataset_id=existing.id)
    print(f"Deleted existing dataset '{dataset_name}'")

dataset = client.create_dataset(dataset_name)
client.create_examples(
    inputs=[e["inputs"] for e in examples],
    outputs=[e["outputs"] for e in examples],
    dataset_id=dataset.id,
)
print(f"Created dataset '{dataset_name}' with {len(examples)} examples")
print(f"View: {dataset.url}")


Deleted existing dataset 'modular-workshops-evals-william-reinhard'
Created dataset 'modular-workshops-evals-william-reinhard' with 4 examples
View: https://smith.langchain.com/o/e3272f09-bf53-4cd5-a37c-93bd272d3078/datasets/92824995-b659-4a34-9de3-0fd7316a72d0


### 2.2 Final-response eval (LLM-as-judge)

<img src="../images/final-response.png" style="width: auto; max-height: 280px; border-radius: 8px;">

Treat the copilot as a black box: did the final response satisfy the analyst's request? 
We'll build the **LLM-as-judge from scratch** so the moving parts are clear:

1. A Pydantic / TypedDict schema for the judge's output (`score`, `reasoning`)
2. A judge prompt that explains the grading criteria
3. `model.with_structured_output(...)` to force the LLM into the schema
4. An evaluator function that calls the judge and returns the score in the shape `client.evaluate` expects


In [7]:
def run_agent_final(inputs: dict) -> dict:
    """Target function: run the copilot and return its final response.

    We also concatenate any files the copilot wrote into the response string so
    the judge can see actual content -- not just the copilot's "saved!" message.
    """
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config=config,
    )

    response = result["messages"][-1].content

    file_dump = []
    for path, file_data in (result.get("files") or {}).items():
        content = file_data
        if isinstance(file_data, dict) and "content" in file_data:
            content = file_data["content"]
        if isinstance(content, list):
            content = "\n".join(content)
        file_dump.append(f"--- {path} ---\n{content}")
    if file_dump:
        response += "\n\nFiles written:\n" + "\n\n".join(file_dump)

    return {"response": response}


In [8]:
from typing_extensions import TypedDict
from langchain_core.messages import SystemMessage, HumanMessage

# Define the judge's output schema -- with_structured_output enforces this shape on the LLM response.
class CorrectnessGrade(TypedDict):
    """Score whether the copilot's response satisfied the analyst's request."""
    score: bool   # True if correct/helpful, False otherwise
    reasoning: str  # one-sentence explanation

# The dataset's `reference_answer` is a SUCCESS RUBRIC, not an expected response text.
# Make that explicit to the judge so it doesn't downscore valid copilot responses that
# happen to be worded differently.
correctness_judge_prompt = """You are an expert grader evaluating a commodities trading copilot's response.

You'll see the analyst's request, the copilot's final response, and a RUBRIC describing what success looks like.

The rubric is NOT the expected response text -- it's the success criteria. Mark `score=True` if the copilot's response demonstrates that the criteria were met (e.g., it confirms the desk note was written, or shows the content that was saved). The copilot's wording doesn't need to match the rubric -- what matters is whether the underlying task was accomplished.

Mark `score=False` only if the response clearly missed the task, contained errors, or refused without good reason.
Give one short sentence of reasoning either way.
"""

# Bind the schema once -- `judge` is now a structured-output LLM.
judge = model.with_structured_output(CorrectnessGrade)

def correctness_evaluator(inputs, outputs, reference_outputs):
    grade = judge.invoke([
        SystemMessage(content=correctness_judge_prompt),
        HumanMessage(content=(
            f"Analyst request: {inputs['query']}\n\n"
            f"Copilot response: {outputs['response']}\n\n"
            f"Success rubric: {reference_outputs['reference_answer']}"
        )),
    ])
    return {"key": "correctness", "score": int(grade["score"]), "comment": grade["reasoning"]}


In [9]:
results = client.evaluate(
    run_agent_final,
    data=dataset_name,
    evaluators=[correctness_evaluator],
    experiment_prefix="final-response",
    max_concurrency=2,
)
print(f"View at: {results.experiment_name}")


View the evaluation results for experiment: 'final-response-fc98aba5' at:
https://smith.langchain.com/o/e3272f09-bf53-4cd5-a37c-93bd272d3078/datasets/92824995-b659-4a34-9de3-0fd7316a72d0/compare?selectedSessions=5246aca3-7b32-47f2-ad3d-955534bd5b96




0it [00:00, ?it/s]

View at: final-response-fc98aba5


### 2.3 Trajectory eval

<img src="../images/trajectory.png" style="width: auto; max-height: 280px; border-radius: 8px;">

Score the **sequence of tool calls** the copilot took, not just the final answer. Three evaluators:

- **`exact_match`** — did it take exactly the right steps in order?
- **`extra_steps`** — how many extra tool calls did it make?
- **`missing_steps`** — how many expected steps did it skip?

`extra_steps` and `missing_steps` use `collections.Counter` for multiset diffs — order doesn't matter for those two, but `exact_match` still catches ordering bugs.


In [10]:
from collections import Counter
from typing import Any

def trajectory_match(outputs, reference_outputs):
    return {
        "key": "exact_match",
        "score": int(outputs["trajectory"] == reference_outputs["trajectory"]),
    }

def extra_steps(outputs, reference_outputs):
    extras = Counter(outputs["trajectory"]) - Counter(reference_outputs["trajectory"])
    return {"key": "extra_steps", "score": sum(extras.values())}

def missing_steps(outputs, reference_outputs):
    missing = Counter(reference_outputs["trajectory"]) - Counter(outputs["trajectory"])
    return {"key": "missing_steps", "score": sum(missing.values())}


Next, we'll define the run function.

In [11]:
def extract_tool_calls(messages: list[Any]) -> list[str]:
    """Extract tool call names from messages in order."""
    tool_names = []
    for msg in messages:
        if getattr(msg, "tool_calls", None):
            tool_names.extend(tc["name"] for tc in msg.tool_calls)
    return tool_names

def run_agent_trajectory(inputs: dict) -> dict:
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config=config,
    )
    return {"trajectory": extract_tool_calls(result["messages"])}


In [12]:
results = client.evaluate(
    run_agent_trajectory,
    data=dataset_name,
    evaluators=[trajectory_match, extra_steps, missing_steps],
    experiment_prefix="trajectory",
    max_concurrency=2,
)
print(f"View at: {results.experiment_name}")


View the evaluation results for experiment: 'trajectory-7c8ea952' at:
https://smith.langchain.com/o/e3272f09-bf53-4cd5-a37c-93bd272d3078/datasets/92824995-b659-4a34-9de3-0fd7316a72d0/compare?selectedSessions=eb4fb588-4785-4637-8239-2aec81577784




0it [00:00, ?it/s]

View at: trajectory-7c8ea952


## Part 3. Online Evaluations

**Online evals** run automatically against every new trace as it lands in your tracing project — same evaluator as in Part 2, just triggered on incoming desk runs instead of a dataset.

LangSmith calls these **run rules**. The Python SDK doesn't expose them directly, so we wrap the REST endpoint with a helper at `utils/langsmith_rules.py`. 
Pass in: a project name, an LLM-as-judge prompt, an output schema. Get back: the rule ID and a deep link to inspect it in the UI.


In [13]:
# Define the LLM-as-judge prompt + schema.
judge_prompt = (
    "You score whether a commodities trading copilot's response satisfied the analyst's request.\n"
    "Reply with correctness (true/false) and one sentence of comment explaining why."
)

judge_schema = {
    "title": "correctness",
    "description": "Score whether the copilot response was correct/helpful.",
    "type": "object",
    "properties": {
        "correctness": {"type": "boolean", "description": "True if the response was correct/helpful"},
        "comment": {"type": "string", "description": "One short sentence explaining the score"},
    },
    "required": ["correctness", "comment"],
    "strict": True,
}

online_rule = create_run_rule(
    client,
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    display_name="workshop-online-correctness",
    sampling_rate=1.0,
    # Score only root traces, not every child LLM/tool/middleware span.
    filter="eq(is_root, true)",
    llm_judge_prompt=judge_prompt,
    llm_judge_schema=judge_schema,
)

print("Rule ID:", online_rule["id"])
print("Open in UI:", online_rule["url"])


Rule ID: 7996fea5-3987-4738-bd47-a34e114dc09f
Open in UI: https://smith.langchain.com/o/e3272f09-bf53-4cd5-a37c-93bd272d3078/evaluators/9f5be813-3ea1-45b1-b3e2-6a543875b3d0?ruleId=7996fea5-3987-4738-bd47-a34e114dc09f&sourceKind=session&sourceId=156d5738-eb06-473c-8924-f83f59eea029


### Evaluating the Trace
Invoke the agent below to see the Evaluator in action!

In [14]:
config = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke({
    "messages": [{"role": "user", "content": "Very lightly research the latest news moving the corn and soybean markets this week"}]
}, config=config)

print(result["messages"][-1].content[:1500])

This week, the corn and soybean markets are being influenced by several key factors:

1. Weather: Warm and dry conditions in the US Corn Belt are creating uncertainty about crop yields. Additionally, developing El Niño conditions may affect global weather patterns, impacting both corn and soybean supplies.

2. Trade: Strong export demand for US corn and firm Chinese demand for soybeans are supporting prices. However, trade tensions including tariffs and geopolitical issues like the Iran conflict and Black Sea disruptions are complicating grain trade flows.

3. Government Policies: The USDA July WASDE report cut US corn ending stocks projections due to strong demand, supporting prices. Biofuel policies also underpin corn demand, while trade policies continue to influence export competitiveness.

4. Demand/Supply: US corn stocks are projected to decline with elevated exports. Soybean demand remains strong owing to growing crush capacity in Brazil and Chinese imports. These factors combin

## Annotation Queues — Close the Loop

Once runs have **feedback scores** (from the online eval above, or any other source), route the low-scoring ones to a desk analyst for review.

LangSmith's annotation queues are that queue. We use **the same `create_run_rule` helper** — this time with `add_to_annotation_queue_id` set instead of an LLM judge. 
Any run matching the filter is added to the queue automatically.

A bare queue just collects runs. To make review consistent across analysts, we give the queue two things:

- **Instructions** (`rubric_instructions`) — free-text guidance shown at the top of the queue: what these runs are, what to look for, how to triage.
- **A feedback rubric** (`rubric_items`) — the structured questions each reviewer answers per run. Each item links a `feedback_key` to a description and score guidance, so scores mean the same thing across the desk. Reviewer scores flow back into LangSmith as feedback — the same feedback you can query, evaluate against, and build future datasets from.


In [15]:
# Reviewer-facing instructions shown at the top of the queue.
rubric_instructions = (
    "These runs are commodities-desk copilot answers flagged by the online "
    "correctness evaluator. For each run, read the analyst's question and the "
    "copilot's answer, then score it against the rubric below. Focus on whether "
    "the answer is factually grounded in the retrieved market data and safe to "
    "put in front of a trader. When something is wrong, leave a note explaining "
    "what a correct answer looks like — those notes become future eval examples."
)

# The structured questions each reviewer answers per run. Each item links a
# feedback_key to guidance so scores mean the same thing across the desk.
rubric_items = [
    {
        "feedback_key": "factually_correct",
        "description": (
            "Is every market claim (prices, balances, supply/demand direction) "
            "accurate and supported by the retrieved data?"
        ),
        "score_descriptions": {
            "0": "Contains a material factual error or unsupported claim",
            "1": "Fully accurate and grounded in the retrieved data",
        },
        "is_required": False,
    },
    {
        "feedback_key": "grounded_in_sources",
        "description": (
            "Did the copilot actually use search/tool results, or did it answer "
            "from memory / hallucinate a source?"
        ),
        "value_descriptions": {
            "grounded": "Answer traces back to retrieved data",
            "partial": "Mix of grounded and unsupported claims",
            "ungrounded": "No supporting retrieval / fabricated source",
        },
        "is_required": False,
    },
    {
        "feedback_key": "trader_ready",
        "description": (
            "Would you send this answer to a trader as-is? Consider clarity, "
            "caveats, and whether it overstates confidence."
        ),
        "score_descriptions": {
            "0": "Not usable — needs a rewrite",
            "1": "Usable with minor edits",
            "2": "Send as-is",
        },
        "is_required": False,
    },
    {
        "feedback_key": "reviewer_notes",
        "description": (
            "Free-text: what was wrong and what a correct answer looks like. "
            "These notes seed future eval examples."
        ),
        "is_required": False,
    },
]

queue = get_or_create_annotation_queue(
    client,
    name=scoped("modular-workshops-needs-review"),
    description="Runs routed here by the workshop's correctness automation rule.",
    rubric_instructions=rubric_instructions,
    rubric_items=rubric_items,
)
print(f"Queue: {queue.name} (id={queue.id})")
print(f"Instructions + {len(rubric_items)} rubric items attached.")


Queue: modular-workshops-needs-review-william-reinhard (id=e4cb0010-4301-480f-8277-d528b166d0ca)
Instructions + 4 rubric items attached.


In [16]:
# Same helper, no evaluator this time -- just a routing rule.
# Filter: only root traces (is_root=true) with correctness > 0.5.
queue_rule = create_run_rule(
    client,
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    display_name="workshop-route-correctness",
    sampling_rate=1.0,
    filter=(
        'and('
        'eq(is_root, true), '
        'eq(feedback_key, "correctness"), '
        'gt(feedback_score, 0.5)'
        ')'
    ),
    add_to_annotation_queue_id=queue.id,
)

print("Queue rule ID:", queue_rule["id"])
print("Open in UI:    ", queue_rule["url"])


Queue rule ID: db2a1f8c-f7ed-4528-a4da-e358ed962727
Open in UI:     https://smith.langchain.com/o/e3272f09-bf53-4cd5-a37c-93bd272d3078/projects/p/156d5738-eb06-473c-8924-f83f59eea029?runview=threads&tab=2


### Trigger both rules

Both rules are live. Run a few more light market questions and you'll see:

1. The online eval fires on each new trace and attaches a `correctness` feedback score (~30s delay).
2. The queue rule fires on each *new feedback* that matches its filter (low correctness) and routes the run to the review queue.


In [17]:
trigger_prompts = [
    "In one sentence, what is the current supply outlook for wheat? Search at most once.",
    "In one sentence, how are weather conditions affecting soybean yields? Search at most once.",
    "In one sentence, what is the demand picture for vegetable oils? Search at most once.",
]

time.sleep(100)

total = 0.0
for q in trigger_prompts:
    cfg = {"configurable": {"thread_id": str(uuid7())}}
    t0 = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=cfg)
    elapsed = time.perf_counter() - t0
    total += elapsed
    print(f"[{elapsed:4.1f}s] {result['messages'][-1].content[:120]}")

# Use the tenant_id LangSmith returned with the rule so the link works regardless of workspace.
tenant_id = queue_rule["payload"]["tenant_id"]

print(f"\nTotal: {total:.1f}s.")
print(f"\nOnline eval rule:  {online_rule['url']}")
print(f"Queue rule:        {queue_rule['url']}")
print(f"Queue (review UI): https://smith.langchain.com/o/{tenant_id}/annotation-queues/{queue.id}")
print("\nFeedback shows up in the rule pages within ~30s; queue placements follow once feedback lands.")


[ 3.3s] The current supply outlook for wheat in 2024 indicates a 10% increase in U.S. production from the previous year due to i
[ 3.7s] Weather conditions such as extreme heat, severe storms, uneven rainfall, and drought during critical growing periods sig
[ 3.2s] The search did not return relevant results about the demand picture for vegetable oils.

Based on general knowledge, the

Total: 10.2s.

Online eval rule:  https://smith.langchain.com/o/e3272f09-bf53-4cd5-a37c-93bd272d3078/evaluators/9f5be813-3ea1-45b1-b3e2-6a543875b3d0?ruleId=7996fea5-3987-4738-bd47-a34e114dc09f&sourceKind=session&sourceId=156d5738-eb06-473c-8924-f83f59eea029
Queue rule:        https://smith.langchain.com/o/e3272f09-bf53-4cd5-a37c-93bd272d3078/projects/p/156d5738-eb06-473c-8924-f83f59eea029?runview=threads&tab=2
Queue (review UI): https://smith.langchain.com/o/e3272f09-bf53-4cd5-a37c-93bd272d3078/annotation-queues/e4cb0010-4301-480f-8277-d528b166d0ca

Feedback shows up in the rule pages within ~30s; queue p

### Common run-rule patterns

Swap the `filter` to build different rules for the desk:

| Use Case | `filter` |
|---|---|
| Low online correctness | `and(eq(feedback_key, "correctness"), lt(feedback_score, 0.5))` |
| Errored runs | `eq(status, "error")` |
| Slow runs | `gt(latency, 10)` |
| Long-running tool calls | `and(eq(run_type, "tool"), gt(latency, 3))` |
| Specific tool fired | `eq(name, "tavily_search")` |

Both rule types — online eval and queue routing — go through the same `create_run_rule` helper. 
Use `delete_run_rule(client, rule_id)` to tear them down when you're done.


## Recap

| Part | What | API |
|---|---|---|
| **Warm-up** | Generate a few traces with the Module 2 trading copilot | `agent.invoke(...)` |
| **1. Tracing + querying** | Auto-capture every desk run; pull back by filter | `LANGSMITH_TRACING=true`, `client.list_runs(filter=...)` |
| **2. Offline evals** | Score on demand against a dataset of desk tasks | `model.with_structured_output(...)` + `client.evaluate` |
| **3. Online evals** | Score every new trace automatically | `create_run_rule(..., llm_judge_prompt=..., llm_judge_schema=...)` |
| **Annotation queues** | Route flagged runs for analyst review | `create_run_rule(..., add_to_annotation_queue_id=...)` |

The full loop: trace → online eval scores it → run rule routes low scores to the queue → analyst reviews → fixes flow into the next dataset.

**Next:** Module 5 — **Engine** automates this entire loop (detect → diagnose → PR → evaluator) on your deployed trading copilot.


## Teardown — Spin Down Your Workshop Resources

Run this once you're done with the workshop to remove the LangSmith resources these modules created and stop any ongoing usage.

The cell below is **idempotent and safe to re-run** — it skips anything that's already gone. It tears down, in order:

1. **Run rules** — the online eval rule and the queue-routing rule (they keep firing on every new trace until deleted).
2. **Annotation queue** — `modular-workshops-needs-review`, including its instructions and rubric.
3. **Eval dataset** — `modular-workshops-evals` from Part 2.

> **Traces** stay in your tracing project as a historical record — delete the project from the LangSmith UI if you want them gone too.

In [18]:
# --- 1. Delete the run rules (stop them firing on new traces) ---
for label, rule in [("online eval", online_rule), ("queue routing", queue_rule)]:
    try:
        delete_run_rule(client, rule["id"])
        print(f"Deleted {label} rule ({rule['id']}).")
    except Exception as e:
        print(f"Skipped {label} rule: {e}")

# --- 2. Delete the annotation queue ---
try:
    client.delete_annotation_queue(queue.id)
    print(f"Deleted annotation queue '{queue.name}' ({queue.id}).")
except Exception as e:
    print(f"Skipped annotation queue: {e}")

# --- 3. Delete the eval dataset ---
if client.has_dataset(dataset_name=dataset_name):
    client.delete_dataset(dataset_name=dataset_name)
    print(f"Deleted dataset '{dataset_name}'.")
else:
    print(f"Dataset '{dataset_name}' already gone.")

print("\nTeardown complete. Traces remain in the tracing project as history.")


Deleted online eval rule (7996fea5-3987-4738-bd47-a34e114dc09f).
Deleted queue routing rule (db2a1f8c-f7ed-4528-a4da-e358ed962727).
Deleted annotation queue 'modular-workshops-needs-review-william-reinhard' (e4cb0010-4301-480f-8277-d528b166d0ca).
Deleted dataset 'modular-workshops-evals-william-reinhard'.

Teardown complete. Traces remain in the tracing project as history.
